In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')


---
## 2. Data Preprocessing

In [2]:
dataset = pd.read_csv('Debit Card Fraud Transaction Prediction.csv', index_col=None)
df = dataset.copy()
print('Shape:', df.shape)
df.head()

Shape: (50000, 21)


,Transaction_ID,User_ID,Transaction_Amount,Transaction_Type,Timestamp,Account_Balance,Device_Type,Location,Merchant_Category,IP_Address_Flag,...,Daily_Transaction_Count,Avg_Transaction_Amount_7d,Failed_Transaction_Count_7d,Card_Type,Card_Age,Transaction_Distance,Authentication_Method,Risk_Score,Is_Weekend,Fraud_Label
0,TXN_33553,USER_1834,39.79,POS,2023-08-14 19:30:00,93213.17,Laptop,Sydney,Travel,0,...,7,437.63,3,Amex,65,883.17,Biometric,0.8494,0,0
1,TXN_9427,USER_7875,1.19,Bank Transfer,2023-06-07 04:01:00,75725.25,Mobile,New York,Clothing,0,...,13,478.76,4,Mastercard,186,2203.36,Password,0.0959,0,1
2,TXN_199,USER_2734,28.96,Online,2023-06-20 15:25:00,1588.96,Tablet,Mumbai,Restaurants,0,...,14,50.01,4,Visa,226,1909.29,Biometric,0.8400,0,1
3,TXN_12447,USER_2617,254.32,ATM Withdrawal,2023-12-07 00:31:00,76807.20,Tablet,New York,Clothing,0,...,8,182.48,4,Visa,76,1311.86,OTP,0.7935,0,1
4,TXN_39489,USER_2014,31.28,POS,2023-11-11 23:44:00,92354.66,Mobile,Mumbai,Electronics,0,...,14,328.69,4,Mastercard,140,966.98,Password,0.3819,1,1


In [3]:
# Basic info
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 50000 entries, 0 to 49999
Data columns (total 21 columns):
 #   Column                        Non-Null Count  Dtype  
---  ------                        --------------  -----  
 0   Transaction_ID                50000 non-null  object 
 1   User_ID                       50000 non-null  object 
 2   Transaction_Amount            50000 non-null  float64
 3   Transaction_Type              50000 non-null  object 
 4   Timestamp                     50000 non-null  object 
 5   Account_Balance               50000 non-null  float64
 6   Device_Type                   50000 non-null  object 
 7   Location                      50000 non-null  object 
 8   Merchant_Category             50000 non-null  object 
 9   IP_Address_Flag               50000 non-null  int64  
 10  Previous_Fraudulent_Activity  50000 non-null  int64  
 11  Daily_Transaction_Count       50000 non-null  int64  
 12  Avg_Transaction_Amount_7d     50000 non-null  float64
 13  F

In [4]:
# Null check
print('Null values:')
print(df.isnull().sum())
print('\nDuplicate rows:', df.duplicated().sum())

Null values:
Transaction_ID                  0
User_ID                         0
Transaction_Amount              0
Transaction_Type                0
Timestamp                       0
Account_Balance                 0
Device_Type                     0
Location                        0
Merchant_Category               0
IP_Address_Flag                 0
Previous_Fraudulent_Activity    0
Daily_Transaction_Count         0
Avg_Transaction_Amount_7d       0
Failed_Transaction_Count_7d     0
Card_Type                       0
Card_Age                        0
Transaction_Distance            0
Authentication_Method           0
Risk_Score                      0
Is_Weekend                      0
Fraud_Label                     0
dtype: int64

Duplicate rows: 0


In [5]:
# Timestamp feature engineering
df['Timestamp'] = pd.to_datetime(df['Timestamp'])
df['Hour']            = df['Timestamp'].dt.hour
df['DayOfWeek']       = df['Timestamp'].dt.dayofweek   # 0=Mon, 6=Sun
df['Month']           = df['Timestamp'].dt.month
df['Is_Night']        = ((df['Hour'] >= 22) | (df['Hour'] <= 5)).astype(int)
df['Is_BusinessHour'] = ((df['Hour'] >= 9)  & (df['Hour'] <= 17)).astype(int)
df.drop(columns=['Timestamp', 'Transaction_ID', 'User_ID'], inplace=True)
print(' Timestamp features created!')
print('New shape:', df.shape)

 Timestamp features created!
New shape: (50000, 23)


In [9]:
# Label Encoding for categorical columns
from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()
cat_cols = ['Transaction_Type','Device_Type','Location',
            'Merchant_Category','Card_Type','Authentication_Method']

for col in cat_cols:
    df[col] = le.fit_transform(df[col])
    print(f'  {col} encoded ')

print('\nFinal shape:', df.shape)

  Transaction_Type encoded 
  Device_Type encoded 
  Location encoded 
  Merchant_Category encoded 
  Card_Type encoded 
  Authentication_Method encoded 

Final shape: (50000, 23)


In [10]:
# Statistical summary
df.describe()

,Transaction_Amount,Transaction_Type,Account_Balance,Device_Type,Location,Merchant_Category,IP_Address_Flag,Previous_Fraudulent_Activity,Daily_Transaction_Count,Avg_Transaction_Amount_7d,...,Transaction_Distance,Authentication_Method,Risk_Score,Is_Weekend,Fraud_Label,Hour,DayOfWeek,Month,Is_Night,Is_BusinessHour
count,50000.000000,50000.000000,50000.000000,50000.000000,50000.000000,50000.000000,50000.00000,50000.000000,50000.000000,50000.000000,...,50000.000000,50000.000000,50000.000000,50000.000000,50000.000000,50000.000000,50000.000000,50000.000000,50000.00000,50000.000000
mean,99.411012,1.503820,50294.065981,1.003960,2.009400,1.999660,0.05020,0.098400,7.485240,255.271924,...,2499.164155,1.498180,0.501556,0.299640,0.321340,11.516380,3.016660,6.527080,0.33056,0.378200
std,98.687292,1.118074,28760.458557,0.816822,1.418038,1.415111,0.21836,0.297858,4.039637,141.382279,...,1442.013834,1.118902,0.287774,0.458105,0.466996,6.902383,2.001165,3.446364,0.47042,0.484943
min,0.000000,0.000000,500.480000,0.000000,0.000000,0.000000,0.00000,0.000000,1.000000,10.000000,...,0.250000,0.000000,0.000100,0.000000,0.000000,0.000000,0.000000,1.000000,0.00000,0.000000
25%,28.677500,1.000000,25355.995000,0.000000,1.000000,1.000000,0.00000,0.000000,4.000000,132.087500,...,1256.497500,0.000000,0.254000,0.000000,0.000000,6.000000,1.000000,4.000000,0.00000,0.000000
50%,69.660000,2.000000,50384.430000,1.000000,2.000000,2.000000,0.00000,0.000000,7.000000,256.085000,...,2490.785000,2.000000,0.502250,0.000000,0.000000,12.000000,3.000000,7.000000,0.00000,0.000000
75%,138.852500,3.000000,75115.135000,2.000000,3.000000,3.000000,0.00000,0.000000,11.000000,378.032500,...,3746.395000,2.000000,0.749525,1.000000,1.000000,17.000000,5.000000,10.000000,1.00000,1.000000
max,1174.140000,3.000000,99998.310000,2.000000,4.000000,4.000000,1.00000,1.000000,14.000000,500.000000,...,4999.930000,3.000000,1.000000,1.000000,1.000000,23.000000,6.000000,12.000000,1.00000,1.000000


In [11]:
# Save preprocessed data
df.to_csv('fraud_preprocessed.csv', index=False)
print(' Saved: fraud_preprocessed.csv')
print('Shape:', df.shape)

 Saved: fraud_preprocessed.csv
Shape: (50000, 23)
